# GQA Base: PyTorch 基础模块与张量操作
本笔记本基于 `rope.py`，把零散代码改成可教学的分段实验。
每一节包含：
- 这个 API 做什么
- 输入张量需要满足什么条件
- 运行后该如何理解输出


### 1. Dropout（输入需为浮点类型）
`nn.Dropout(p)` 会在训练时随机把一部分元素置零，并按比例缩放剩余元素。
- `p=0.5` 表示约 50% 概率被置零
- Dropout **只接受浮点张量**，不能直接对 `LongTensor` 做随机置零
- 推理模式（`model.eval()`）下 Dropout 不会随机丢弃


In [3]:
import torch
import torch.nn as nn

dropout_layer = nn.Dropout(p=0.5)
t = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
print(dropout_layer(t))


tensor([0., 4., 6.])


### 2. 线性层 `nn.Linear`
线性层实现仿射变换：`y = xW^T + b`。
- 这里 `in_features=3`，所以输入最后一维必须是 3
- `out_features=5`，所以输出最后一维变为 5
- 常用于把特征投影到新维度（例如注意力中的 Q/K/V 投影）


In [4]:
t = torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)
linear_layer = nn.Linear(in_features=3, out_features=5, bias=True)
print(linear_layer(t))


tensor([[ 0.4650,  1.4852, -1.3808, -0.7884,  0.4669]],
       grad_fn=<AddmmBackward0>)


### 3. 张量视图变换 `view`
`view` 在**元素总数不变**时重解释张量形状。
- 原张量共 `2x6=12` 个元素
- 可以改成 `(3,4)` 或 `(4,3)`，本质是同一块数据不同视图
- 若内存不连续，通常先 `contiguous()` 再 `view()`


In [6]:
t = torch.tensor([[1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12]])

t_view1 = t.view(3, 4)
print(t_view1)
t_view2 = t.view(4, 3)
print(t_view2)


tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])


### 4. 上三角矩阵 `torch.triu`
`torch.triu(x, diagonal=k)` 保留上三角，其余位置置零。
- `diagonal=0`：保留主对角线及以上
- `diagonal=1`：主对角线也会被置零
- 在注意力机制里常用于构造因果 Mask（屏蔽未来位置信息）


In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(torch.triu(x, diagonal=0))
print(torch.triu(x, diagonal=1))


### 5. 形状重塑 `torch.reshape`
`reshape` 会在可能时返回视图，不行时返回拷贝。
- `(2,3)` 明确指定新形状
- `(3,-1)` 中 `-1` 表示自动推断该维大小
- 使用前后元素总数必须一致


In [ ]:
x = torch.arange(1, 7)
y = torch.reshape(x, (2, 3))
print(y)
z = torch.reshape(x, (3, -1))
print(z)
